# Phase 2: Advanced Stats Calculator
Calculate rate stats, park factors, league adjustments, OPS+, ERA-, FIP-, FIP.

In [1]:
import sys
sys.path.insert(0, '..')

from src.advanced_stats import *
from src.data_builder import load_config

cfg = load_config()

In [2]:
# Build advanced batter stats
batters = build_advanced_batters(cfg)
print(f'Shape: {batters.shape}')
print(f'New columns: {[c for c in batters.columns if c not in ["id","year","G"] and not c.startswith("b_")]}')
print(f'\nOPS+ league average check (should be ~100):')
qual = batters[~batters['LOW_SAMPLE']]
print(qual.groupby('year')['OPS_plus'].mean())

Shape: (15521, 42)
New columns: ['last', 'first', 'team', 'primary_pos', 'LOW_SAMPLE', 'TB', 'AVG', 'OBP', 'SLG', 'OPS', 'ISO', 'BABIP', 'K_pct', 'BB_pct', 'HR_PA', 'BB_K_pct', 'SB_pct', 'PF', 'lg_OBP', 'lg_SLG', 'OPS_plus']

OPS+ league average check (should be ~100):
year
2015    104.573256
2016    103.397482
2017    105.367433
2018    103.774093
2019    104.819434
2020    100.136849
2021    104.711290
2022    101.724525
2023    101.094548
2024     96.881018
2025    100.866406
Name: OPS_plus, dtype: float64


In [3]:
# Build advanced pitcher stats
pitchers = build_advanced_pitchers(cfg)
print(f'Shape: {pitchers.shape}')
print(f'\nERA- league average check (should be ~100):')
qual_p = pitchers[~pitchers['LOW_SAMPLE']]
print(qual_p.groupby('year')['ERA_minus'].mean())
print(f'\nFIP- league average check:')
print(qual_p.groupby('year')['FIP_minus'].mean())
print(f'\nFIP constant per year (should be ~3.10):')
print(qual_p.groupby('year')['FIP_constant'].first())

Shape: (8968, 45)

ERA- league average check (should be ~100):
year
2015    95.850722
2016    95.926121
2017    95.604221
2018    94.299474
2019    96.133439
2020    93.221129
2021    93.991834
2022    95.547625
2023    96.392738
2024    92.636496
2025    92.219263
Name: ERA_minus, dtype: float64

FIP- league average check:
year
2015    97.655938
2016    97.345588
2017    96.749187
2018    96.219277
2019    96.808019
2020    96.387141
2021    97.093012
2022    96.341027
2023    98.060207
2024    93.246003
2025    94.845227
Name: FIP_minus, dtype: float64

FIP constant per year (should be ~3.10):
year
2015    3.126752
2016    3.138871
2017    3.150149
2018    3.156533
2019    3.202817
2020    3.182110
2021    3.162431
2022    3.106163
2023    3.249192
2024    3.159591
2025    3.128033
Name: FIP_constant, dtype: float64


In [4]:
# Park factor verification
pf = calculate_park_factors(cfg)
print('Top 5 park factors (2023):')
pf_2023 = pf[pf['year']==2023].sort_values('PF', ascending=False)
print(pf_2023[['team','site','PF']].head())
print(f'\nCoors Field PF:', pf_2023[pf_2023['site'].str.contains('DEN', na=False)]['PF'].values)

Top 5 park factors (2023):
     team   site        PF
2880  COL  DEN02  1.289712
2875  BOS  BOS07  1.162734
2873  ATL  ATL03  1.072106
2871  ANA  ANA01  1.066445
2878  CIN  CIN09  1.056108

Coors Field PF: [1.2897115]


In [5]:
# Spot checks
judge22 = batters[(batters['last']=='Judge') & (batters['year']==2022)]
if len(judge22):
    print(f'Judge 2022: AVG={judge22["AVG"].values[0]:.3f}, OPS={judge22["OPS"].values[0]:.3f}, OPS+={judge22["OPS_plus"].values[0]:.0f}')

cole23 = pitchers[(pitchers['last']=='Cole') & (pitchers['year']==2023)]
if len(cole23):
    print(f'Cole 2023: ERA={cole23["ERA"].values[0]:.2f}, FIP={cole23["FIP"].values[0]:.2f}, ERA-={cole23["ERA_minus"].values[0]:.0f}, FIP-={cole23["FIP_minus"].values[0]:.0f}')

# Verify ERA_plus is gone
assert 'ERA_plus' not in pitchers.columns, "ERA_plus should be removed!"
print('\nConfirmed: ERA_plus removed, ERA_minus and FIP_minus present')

Judge 2022: AVG=0.311, OPS=1.108, OPS+=207
Cole 2023: ERA=2.63, FIP=3.16, ERA-=62, FIP-=75

Confirmed: ERA_plus removed, ERA_minus and FIP_minus present
